In [4]:
!pip install pyswarm

  Using cached pyswarm-0.6.tar.gz (4.3 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyswarm: filename=pyswarm-0.6-py3-none-any.whl size=4518 sha256=5ea29c0212a9ab4cdf6c5310186dff9b998b3166aa871bb09dad621799bb29f5
  Stored in directory: /Users/madsbertelsen/Library/Caches/pip/wheels/40/f3/da/00b82a03b46209e55182b57412ec1d6b21c9795ead6b048313
Successfully built pyswarm


In [ ]:
from pyswarm import pso  # may need to restart notebook

In [7]:
import matplotlib.pyplot as plt
import mcstasscript as ms

instr = ms.McStas_instr("test")
source = instr.add_component("source", "Source_simple")
source.set_parameters(
    xwidth=0.1,
    yheight=0.1,
    focus_xw=0.02,
    focus_yh=0.03,
    dist=10,
    flux=instr.add_parameter("flux", value=100),
    lambda0=instr.add_parameter("wavelength", value=4.0),
    dlambda=instr.add_parameter("delta_wavelength", value=2.0),
)

sample_position = instr.add_component("sample_position", "Arm")
sample_position.set_AT(source.dist, RELATIVE=source)


sample = instr.add_component("sample", "Res_sample", RELATIVE=sample_position)
sample.set_parameters(
    radius=0.01, yheight=0.03,
    target_x=0, target_y=0, target_z=1, focus_aw=2*175 + 1, focus_ah=30,
    E0=5, dE=0.01, 
)


detector_index = 0
pixel_min = 0

detector_direction = instr.add_component(
    "detector_direction_banana_1",
    "Arm",
    RELATIVE=sample_position,
    ROTATED=[0, 0, 0],
)

xbins = 20
ybins = 12
monitor = instr.add_component("Banana_1", "Monitor_nD")
monitor.set_parameters(
    radius=1.0,
    yheight=0.5,
    filename='"direct_event_banana_signal.dat"',
    restore_neutron=1,
)
monitor.options = (
    f'"banana theta bins={xbins} limits=[5, 175] '
    f'y bins={ybins}"'
)

monitor.set_AT(0.0, RELATIVE=detector_direction)


instrument = instr

In [10]:
from pyswarm import pso

def starter_fom(data):
    detector_data = ms.name_search("Banana_1", data)
    return -detector_data.metadata.total_I

def simulate(x, par_names=None, instrument=None, fom=starter_fom):
    # Need to ensure parameters are given
    if par_names is None:
        raise ValueError("No parameter names specified.")

    for par, par_value in zip(par_names, x):
        par_dict = {}
        par_dict[par] = par_value

        instrument.set_parameters(par_dict)

    # Run simulation
    data = instrument.backengine()

    # Verbose mode
    print(x, "\t", data[0])

    return fom(data)


# Set some instrument settings
instrument.write_full_instrument()
instrument.settings(ncount=1E6, mpi = 6, force_compile=False, suppress_output=True)

param_names = ["flux"] # input parameter names to optimize here as list
lb = [10] # lower bounds
ub = [100] # upper bounds

# Make a dict with the keyward arguments needed by the optimized function
kwargs_input = dict(par_names=param_names, instrument=instrument, fom=starter_fom)

# perform optimization
xopt, fopt = pso(simulate, lb, ub, swarmsize=3, maxiter=10, kwargs=kwargs_input)

[62.3723684] 	 McStasData: Banana_1 type: 2D  I:0.0532663 E:8.2263e-05 N:453059.0
[62.4180104] 	 McStasData: Banana_1 type: 2D  I:0.053358 E:8.23566e-05 N:453694.0
[11.34528408] 	 McStasData: Banana_1 type: 2D  I:0.00969124 E:1.49689e-05 N:452804.0
[35.78853271] 	 McStasData: Banana_1 type: 2D  I:0.0305572 E:4.71975e-05 N:453054.0
[21.39216938] 	 McStasData: Banana_1 type: 2D  I:0.0182262 E:2.81769e-05 N:452196.0
[10.] 	 McStasData: Banana_1 type: 2D  I:0.0085344 E:1.31826e-05 N:453069.0
[28.32173888] 	 McStasData: Banana_1 type: 2D  I:0.0241619 E:3.73357e-05 N:452541.0
[22.88596937] 	 McStasData: Banana_1 type: 2D  I:0.0195736 E:3.02045e-05 N:453915.0
[12.12461347] 	 McStasData: Banana_1 type: 2D  I:0.0103489 E:1.5987e-05 N:452707.0
[49.9927446] 	 McStasData: Banana_1 type: 2D  I:0.0426933 E:6.59347e-05 N:453177.0
[48.66441764] 	 McStasData: Banana_1 type: 2D  I:0.0415587 E:6.4183e-05 N:452992.0
[26.19995727] 	 McStasData: Banana_1 type: 2D  I:0.0223795 E:3.45542e-05 N:453231.0
[69.24

In [12]:
xopt, fopt

(array([94.32615156]), -0.0804737)